# NYC Essential Workers: 'Essential', But For Whom?
### COVID-19 Mortality, Transit Dependence, and Neighborhood Income Across New York City, 2020–2022

**Student:** Daniel Foulen  

**Class:** IS 380 

**Date:** 5/20/2026


**Data:** `../data/merged_modzcta.geojson` — 177 MODZCTAs, WGS84

---

New York City's subway system ran throughout COVID-19. There was no passenger shutdown. Office workers could stay home with the remote-work mandates, but not every New Yorker had the option to stop riding. 

Essential workers in health care, food service, building maintenance, and transit itself kept going. This project asks whether the geography of that transit dependence overlapped with the geography of COVID-19 burden.

Three public datasets are joined to 177 Modified ZIP Code Tabulation Areas (MODZCTAs), the neighborhood geography used by NYC Health for disease surveillance:

- MTA Subway Hourly Ridership via NY Open Data (2020–2022)
- NYC Department of Health cumulative COVID-19 outcomes by neighborhood
- U.S. Census Bureau ACS 2021 median income and poverty estimates

The maps trace the story from economic baseline to mortality outcome to spatial structure.

| # | File | Description |
|---|------|-------------|
| 1 | `map_income_tier.html` | Income tier choropleth with toggleable subway lines and stations |
| 2 | `map_covid_death.html` | COVID-19 death rate choropleth, natural breaks (Jenks), white to blood-red gradient |
| 3 | `map_lisa.html` | LISA spatial cluster map, Queen contiguity, p < 0.05 |


> `lisa_type` is not stored in the GeoJSON and is recomputed in the LISA cell below before any map is built.


In [7]:
import geopandas as gpd
import pandas as pd
import json as _json
import numpy as np
import folium
from folium.features import GeoJsonTooltip
from libpysal.weights import Queen
from esda.moran import Moran_Local
import mapclassify
import warnings

## Data and Methodology

Three public datasets are joined to a shared neighborhood geography: MTA Subway Hourly Ridership (NY Open Data, 2020–2022), NYC Department of Health cumulative COVID-19 outcomes by MODZCTA, and U.S. Census Bureau ACS 2021 5-Year Estimates for median household income and poverty rates.

**Geography.** All three datasets are joined to the Modified ZIP Code Tabulation Area (MODZCTA), with 177 residential neighborhoods defined by NYC Health for disease surveillance. Census data is reported at the ZCTA level. A NYC Health crosswalk maps 214 ZCTAs to their corresponding 177 MODZCTAs, aggregating income using a population-weighted average and poverty rate by summing numerators and denominators separately.

**Ridership.** The MTA source contains over 120 million hourly station-level records. A server-side SoQL query aggregates these to monthly totals per station complex before transmission, reducing the dataset to ~17,500 rows. Each of the 428 station complexes is assigned to a MODZCTA via spatial join (point-in-polygon). Five stations bordering Central Park fell outside all residential polygons and were assigned by nearest-neighbor distance. 54 MODZCTAs have no assigned subway station and carry null ridership values throughout the analysis. Annual ridership totals and percent change from 2020 to 2022 are computed per MODZCTA.

**Income tier.** Neighborhoods are classified into three tiers using fixed NYC-specific thresholds: working poor (below $50K), working class ($50K–$100K), upper-middle class (above $100K). These reflect material differences in NYC cost of living rather than equal-sized statistical bins.

**A note on case counts.** COVID-19 case counts are present in the dataset but not used as a primary variable. In 2020, testing access was uneven. Communities with historically grounded distrust of medical institutions were less likely to seek testing, meaning case counts in working poor outer-borough neighborhoods are likely undercounts relative to their actual burden. Death rate is a more defensible measure, and the income-death rate gradient found here is probably conservative as a result.

**Output.** All sources are merged into a single file, `merged_modzcta.geojson`, 177 features, 17 columns, WGS84. This is the sole data source for all three maps.

| Column | Source | Used in |
|--------|--------|---------|
| `COVID_DEATH_RATE` | NYC DOHMH | Maps 1, 2, 3 |
| `income_tier` | ACS 2021 via Census API | Maps 1, 2, 3 |
| `median_income_wtd` | ACS 2021 | Maps 1, 2, 3 |
| `poverty_rate` | ACS 2021 | Maps 1, 2, 3 |
| `ridership_2020` | MTA via NY Open Data | Maps 1, 2, 3 |
| `pct_change_20_22` | MTA via NY Open Data | Maps 1, 2, 3 |
| `lisa_type` | Recomputed in IS 380 notebook | Map 3 |
| `geometry` | NYC DOHMH MODZCTA boundaries | All maps |

**What was excluded.** COVID case counts and case rates are present in the GeoJSON but not used as primary map variables. Ridership for 2021 is retained in the file but not mapped. The rebound metric measures climb from the 2020 floor, not recovery to a pre-pandemic baseline. **Pre-2020 ridership data was not incorporated for technical scope reasons.**

In [8]:
DATA_PATH = '../data/merged_modzcta.geojson'

gdf = gpd.read_file(DATA_PATH)
assert len(gdf) >= 170, f'Expected >= 170 features, got {len(gdf)}'

gdf.shape

(177, 17)

## Technical Notes

### Spatial Autocorrelation (LISA)

`lisa_type` is not stored in the GeoJSON and must be recomputed 
before any map is built. The cell below runs Local Moran's I on 
`COVID_DEATH_RATE` using Queen contiguity weights 
(row-standardized, seed = 42) and assigns cluster types at 
p < 0.05.

| Quadrant | Label | Meaning |
|----------|-------|---------|
| 1 | High-High | High rate, high-rate neighbors |
| 2 | Low-High | Low rate, high-rate neighbors |
| 3 | Low-Low | Low rate, low-rate neighbors |
| 4 | High-Low | High rate, low-rate neighbors |

Three columns are added to `gdf` for use in the Map 3 tooltip: 
`local_i` (Local Moran's I statistic), `p_value`, and 
`neighbor_names` (sorted, deduplicated list of contiguous 
neighborhood names).

> **Known island:** Roosevelt Island has no Queen-contiguous polygon neighbors and is excluded from spatial lag calculations.


> East New York (1,612/100K, the highest in the dataset) does not appear in the High-High cluster due to its limited contiguous neighbors at the southeastern city boundary.

### Map Construction

Each map is built as a folium Figure object, rendered inline in 
the notebook, and saved as a standalone HTML file for iframe 
embedding in ArcGIS Story Maps.

Subway line geometries are fetched client-side at render time 
from the MTA Open Data API rather than embedded inline. A 
previous version of this analysis embedded all 29 subway line 
layers directly into the HTML output, producing a ~10MB file. 
The current approach injects a JavaScript fetch() call instead, 
keeping each HTML file under 3MB and suitable for Story Maps 
embedding without degrading interactivity.

Station markers are loaded from the local 
`stations_with_modzcta.csv` file and added as CircleMarker 
objects. Display columns (formatted income, death rate, 
ridership, poverty rate) are pre-computed before the GeoJson 
layer is built so the style and tooltip functions read cleanly 
from feature properties.

All three maps use CartoDB positron as the basemap and 
GeoJsonTooltip with consistent field structure and null handling 
across all layers.

In [9]:
# Build Queen weights
w = Queen.from_dataframe(gdf, use_index=True, silence_warnings=True)
w.transform = 'r'

# Run Local Moran's I on COVID_DEATH_RATE
y = gdf['COVID_DEATH_RATE'].values
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    lisa = Moran_Local(y, w, seed=42)

# Assign lisa_type
LISA_LABELS = {1: 'High-High', 2: 'Low-High', 3: 'Low-Low', 4: 'High-Low'}
sig = lisa.p_sim < 0.05
gdf['lisa_type'] = [
    LISA_LABELS[lisa.q[i]] if sig[i] else 'Not Significant'
    for i in range(len(gdf))
]

# Local Moran's I statistic and p-value
gdf['local_i'] = np.round(lisa.Is, 4)
gdf['p_value'] = np.round(lisa.p_sim, 4)

# Neighbor names for tooltip
idx_to_name = gdf['NEIGHBORHOOD_NAME'].to_dict()
neighbor_names = []
for i in range(len(gdf)):
    neighbors = w.neighbors[i]
    if not neighbors:
        neighbor_names.append('No contiguous neighbors')
    else:
        names = sorted(set(idx_to_name[j] for j in neighbors))
        neighbor_names.append(', '.join(names))
gdf['neighbor_names'] = neighbor_names

# Cluster type counts
gdf['lisa_type'].value_counts().rename('neighborhoods').to_frame()

,neighborhoods
lisa_type,
Not Significant,135
Low-Low,25
High-High,9
Low-High,6
High-Low,2


## Map 1: Income Tier Choropleth

Primary fill color is income tier (working poor / working class / 
upper-middle class). Subway lines and station markers are 
toggleable layers fetched live from the MTA Open Data API. The 
tooltip surfaces median income, COVID death rate, ridership, 
poverty rate, LISA cluster type, and assigned station names for 
each neighborhood.

Saved to `map_income_tier.html`.

In [10]:
PALETTE = {
    'working poor':       '#FECDAA',
    'working class':      '#B2F7EF',
    'upper-middle class': '#05668D',
}

LINE_COLORS = {
    'A': '#0039A6', 'C': '#0039A6', 'E': '#0039A6',
    'B': '#FF6319', 'D': '#FF6319', 'F': '#FF6319', 'M': '#FF6319',
    'G': '#6CBE45',
    'J': '#996633', 'Z': '#996633',
    'L': '#A7A9AC',
    'N': '#FCCC0A', 'Q': '#FCCC0A', 'R': '#FCCC0A', 'W': '#FCCC0A',
    '1': '#EE352E', '2': '#EE352E', '3': '#EE352E',
    '4': '#00933C', '5': '#00933C', '6': '#00933C',
    '7': '#B933AD',
    'S': '#808183',
}

LISA_DISPLAY = {
    'High-High':       'High burden zone',
    'Low-Low':         'Low burden zone',
    'High-Low':        'Outlier (high in low-burden area)',
    'Low-High':        'Outlier (low in high-burden area)',
    'Not Significant': 'No significant clustering',
}

# ── Display columns ──────────────────────────────────────────────────────
gdf_map1 = gdf.to_crs(epsg=4326).copy()
gdf_map1['color']                  = gdf_map1['income_tier'].map(PALETTE).fillna('#aaaaaa')
gdf_map1['lisa_display']           = gdf_map1['lisa_type'].map(LISA_DISPLAY)
gdf_map1['median_income_display']  = gdf_map1['median_income_wtd'].apply(
    lambda x: 'N/A' if pd.isna(x) else f'${x:,.0f}'
)
gdf_map1['covid_death_rate_display'] = gdf_map1['COVID_DEATH_RATE'].apply(
    lambda x: 'N/A' if pd.isna(x) else f'{x:,.1f}'
)
gdf_map1['ridership_2020_display'] = gdf_map1['ridership_2020'].apply(
    lambda x: 'No assigned station' if pd.isna(x) else f'{x:,.0f}'
)
gdf_map1['pct_change_display']     = gdf_map1['pct_change_20_22'].apply(
    lambda x: 'N/A' if pd.isna(x) else f'{x * 100:.1f}%'
)
gdf_map1['poverty_rate_display']   = gdf_map1['poverty_rate'].apply(
    lambda x: 'N/A' if pd.isna(x) else f'{x * 100:.1f}%'
)

# Join station names to each MODZCTA
stations_df = pd.read_csv('../data/stations_with_modzcta.csv')
stations_agg = (
    stations_df.groupby('MODZCTA')['station_complex']
    .apply(lambda names: ', '.join(sorted(set(names))))
    .reset_index()
    .rename(columns={'station_complex': 'stations'})
)
stations_agg['MODZCTA'] = stations_agg['MODZCTA'].astype(int)
gdf_map1['MODZCTA_int'] = gdf_map1['MODZCTA'].astype(int)
gdf_map1 = gdf_map1.merge(stations_agg, left_on='MODZCTA_int', right_on='MODZCTA', how='left')
gdf_map1['stations'] = gdf_map1['stations'].fillna('No assigned station')
gdf_map1 = gdf_map1.drop(columns=['MODZCTA_int', 'MODZCTA_y'], errors='ignore')
if 'MODZCTA_x' in gdf_map1.columns:
    gdf_map1 = gdf_map1.rename(columns={'MODZCTA_x': 'MODZCTA'})

# ── Build map ─────────────────────────────────────────────────────────────
fig1 = folium.Figure(width='100%', height=650)
m1 = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles='CartoDB positron',
).add_to(fig1)

def style_fn1(feature):
    return {
        'fillColor': feature['properties']['color'],
        'color': 'white',
        'weight': 0.6,
        'fillOpacity': 0.72,
    }

def highlight_fn1(feature):
    return {
        'fillColor': feature['properties']['color'],
        'color': '#333333',
        'weight': 2.5,
        'fillOpacity': 0.88,
    }

tooltip1 = GeoJsonTooltip(
    fields=[
        'NEIGHBORHOOD_NAME', 'MODZCTA', 'BOROUGH_GROUP',
        'income_tier', 'median_income_display',
        'covid_death_rate_display', 'ridership_2020_display',
        'pct_change_display', 'poverty_rate_display',
        'lisa_display', 'stations',
    ],
    aliases=[
        'Neighborhood', 'MODZCTA', 'Borough',
        'Income tier', 'Median income',
        'COVID deaths / 100K', '2020 ridership',
        'Ridership rebound 2020–22', 'Poverty rate',
        'LISA cluster', 'Subway stations',
    ],
    localize=False,
    sticky=True,
    style='background-color: #ffffff; border: 1px solid #999; border-radius: 4px; box-shadow: 3px 3px 6px rgba(0,0,0,0.25); font-size: 13px; max-width: 420px; white-space: normal;',
)

folium.GeoJson(
    gdf_map1.__geo_interface__,
    style_function=style_fn1,
    highlight_function=highlight_fn1,
    tooltip=tooltip1,
    name='Neighborhoods',
).add_to(m1)

# ── Subway lines (client-side JS fetch via MacroElement) ──────────────────
from branca.element import MacroElement
from jinja2 import Template
import json as _json

class _SubwayLayer1(MacroElement):
    def __init__(self, line_colors):
        super().__init__()
        self.line_colors_json = _json.dumps(line_colors)
        self._template = Template(
            "{%- macro script(this, kwargs) -%}(function(){var _map={{this._parent.get_name()}};_map.createPane('subwayPane');_map.getPane('subwayPane').style.zIndex='450';setTimeout(function(){fetch('https://data.ny.gov/resource/s692-irgq.geojson?$limit=50000').then(function(r){return r.json();}).then(function(data){var lc={{this.line_colors_json}};L.geoJson(data,{pane:'subwayPane',style:function(f){var s=(f.properties.service||'').trim().toUpperCase();return {color:lc[s]||'#555555',weight:3,opacity:0.8};},onEachFeature:function(f,l){if(f.properties.service_name)l.bindTooltip(f.properties.service_name);}}).addTo(_map);}).catch(function(e){console.warn('Subway fetch failed:',e);});},1000);})();{%- endmacro -%}"
        )

_SubwayLayer1(LINE_COLORS).add_to(m1)


class _StationPane(MacroElement):
    def __init__(self):
        super().__init__()
        self._template = Template(
            "{%- macro script(this, kwargs) -%}"
            "{{this._parent.get_name()}}.createPane('stationsPane');"
            "{{this._parent.get_name()}}.getPane('stationsPane').style.zIndex='550';"
            "{%- endmacro -%}"
        )
_StationPane().add_to(m1)

# ── Station markers ───────────────────────────────────────────────────────
stations_fg = folium.FeatureGroup(name='Subway stations', show=True)
for _, row in stations_df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='#333333',
        fill=True,
        fill_color='#ffffff',
        fill_opacity=0.9,
        weight=1.5,
        tooltip=row['station_complex'],
    ).add_to(stations_fg)

# ── Hover effects ─────────────────────────────────────────────────────────
class _StationHover(MacroElement):
    def __init__(self):
        super().__init__()
        self._template = Template(
            "{%- macro script(this, kwargs) -%}(function(){var _fg={{this._parent.get_name()}};_fg.eachLayer(function(l){l.on('mouseover',function(){this.setRadius(6);this.setStyle({fillColor:'#FCCC0A',fillOpacity:1,color:'#1a1a1a',weight:2.5});});l.on('mouseout',function(){this.setRadius(4);this.setStyle({fillColor:'#ffffff',fillOpacity:0.9,color:'#333333',weight:1.5});});});})();{%- endmacro -%}"
        )
_StationHover().add_to(stations_fg)
stations_fg.add_to(m1)

# ── Legend ────────────────────────────────────────────────────────────────
legend_html1 = (
    '<div style="position:fixed; bottom:30px; left:30px; z-index:1000; '
    'background:white; padding:12px 16px; border-radius:6px; '
    'box-shadow:0 2px 6px rgba(0,0,0,0.3); font-family:sans-serif; '
    'font-size:13px; line-height:1.8;">'
    '<b>Income Tier</b><br>'
    '<span style="background:#FECDAA;width:14px;height:14px;'
    'display:inline-block;border-radius:2px;margin-right:6px;"></span>Working poor<br>'
    '<span style="background:#B2F7EF;width:14px;height:14px;'
    'display:inline-block;border-radius:2px;margin-right:6px;"></span>Working class<br>'
    '<span style="background:#05668D;width:14px;height:14px;'
    'display:inline-block;border-radius:2px;margin-right:6px;"></span>Upper-middle class<br>'
    '<span style="background:#aaaaaa;width:14px;height:14px;'
    'display:inline-block;border-radius:2px;margin-right:6px;"></span>No data'
    '</div>'
)
m1.get_root().html.add_child(folium.Element(legend_html1))

folium.LayerControl(collapsed=False).add_to(m1)

# ── Save HTML and inject stationsPane into both the file and inline display ─
import re as _re

m1.save('map_income_tier.html')
with open('map_income_tier.html') as _f: _html = _f.read()
_html = _re.sub(
    r'(L\.circleMarker\(\s*\[[^\]]+\],\s*){',
    r'\1{"pane": "stationsPane", ',
    _html
)
with open('map_income_tier.html', 'w') as _f: _f.write(_html)

from IPython.display import IFrame as _IFrame
_IFrame("map_income_tier.html", width="100%", height=650)

### Map 1: Story Map Narrative

**Neighborhood Income Tier and Subway Infrastructure,
New York City (2020–2022)**

This map operates at the MODZCTA level with 177 residential 
neighborhoods across five boroughs. This is the finest geography 
available in NYC public health data as COVID-19 outcomes are only 
reported at this resolution.

Income tier is represented as a categorical choropleth. The 
three tiers, **working poor, working class, and upper-middle 
class**, are discrete classifications, not points on a 
continuous scale. A sequential gradient would imply ordering 
that does not exist between them.

The tier thresholds are fixed at $50,000 and $100,000 in median 
household income. These are not equal-sized statistical bins, 
they reflect real material differences in New York City's cost 
of living. Working poor neighborhoods ($21K–$50K) represent 16% 
of the city's residential geography. Working class ($50K–$100K) 
is the majority at 56%. Upper-middle class ($100K+) accounts for 
27%.

The pattern is legible immediately. Upper-middle class 
neighborhoods concentrate in the Manhattan core and along 
waterfront areas in Brooklyn and Queens. Working poor 
neighborhoods sit at the outer edges, like East New York, Cypress 
Hills, and the Brighton Beach corridor in Brooklyn; Far Rockaway 
and Jamaica in Queens; scattered across the Bronx. Working class 
neighborhoods fill the geography in between.

The subway overlay is the analytical layer that connects this 
map to the ones that follow. The lines thread through all three 
income tiers, but they serve different kinds of trips in each. 
In upper-middle class neighborhoods, they connect to office jobs 
that could go remote. In working poor and working class 
neighborhoods, they connect to jobs that could not. This map 
sets the baseline. 

Before showing who died, it shows who lives where, and how they move.

--------

## Map 2: COVID Death Rate Choropleth

Cumulative COVID-19 deaths per 100,000 residents (through 2022) 
at the MODZCTA level. Classification is **natural breaks 
(Jenks), k = 5**; color runs from white (#ffffff, lowest burden) 
to dark blood red (#8b0000, highest burden). The 54 MODZCTAs 
with no assigned subway station display "No assigned station" / 
"N/A" in ridership fields.

Saved to `map_covid_death.html`.

In [11]:
# ── Classification ──────────────────────────────────────────────────────
nb = mapclassify.NaturalBreaks(gdf['COVID_DEATH_RATE'].dropna(), k=5)
min_val = float(gdf['COVID_DEATH_RATE'].min())
bin_edges = [min_val] + list(nb.bins)

# ── Color gradient: white (#ffffff) to blood red (#8b0000) ────────────────
def lerp_color(t):
    r = int(255 + (0x8b - 255) * t)
    g = int(255 * (1 - t))
    b = int(255 * (1 - t))
    return f'#{r:02x}{g:02x}{b:02x}'

BIN_COLORS = [lerp_color(i / 4) for i in range(5)]

def darken_hex(h, f=0.75):
    h = h.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'#{int(r * f):02x}{int(g * f):02x}{int(b * f):02x}'

def get_bin_color(rate):
    if pd.isna(rate):
        return '#cccccc'
    for i, upper in enumerate(nb.bins):
        if rate <= upper:
            return BIN_COLORS[i]
    return BIN_COLORS[-1]

# ── Pre-format display columns ────────────────────────────────────────────
gdf_map2 = gdf.to_crs(epsg=4326).copy()
gdf_map2['bin_color']           = gdf_map2['COVID_DEATH_RATE'].apply(get_bin_color)
gdf_map2['dark_bin_color']      = gdf_map2['bin_color'].apply(darken_hex)
gdf_map2['covid_death_rate_fmt'] = gdf_map2['COVID_DEATH_RATE'].apply(
    lambda x: 'N/A' if pd.isna(x) else f'{x:.1f}'
)
gdf_map2['poverty_rate_fmt']    = gdf_map2['poverty_rate'].apply(
    lambda x: 'N/A' if pd.isna(x) else f'{x:.1%}'
)
gdf_map2['ridership_2020_fmt']  = gdf_map2['ridership_2020'].apply(
    lambda x: 'No assigned station' if pd.isna(x) else f'{x:,.0f}'
)
gdf_map2['pct_change_fmt']      = gdf_map2['pct_change_20_22'].apply(
    lambda x: 'N/A' if pd.isna(x) else f'{x:.1%}'
)

# ── Build map ─────────────────────────────────────────────────────────────
fig2 = folium.Figure(width='100%', height=620)
m2 = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles='CartoDB positron',
).add_to(fig2)

def style_fn2(feature):
    return {
        'fillColor': feature['properties']['bin_color'],
        'color': 'white',
        'weight': 0.6,
        'fillOpacity': 0.75,
    }

def highlight_fn2(feature):
    return {
        'fillColor': feature['properties']['dark_bin_color'],
        'color': '#333333',
        'weight': 2.5,
        'fillOpacity': 0.90,
    }

tooltip2 = GeoJsonTooltip(
    fields=[
        'NEIGHBORHOOD_NAME', 'BOROUGH_GROUP',
        'covid_death_rate_fmt', 'COVID_DEATH_COUNT',
        'income_tier', 'poverty_rate_fmt',
        'ridership_2020_fmt', 'pct_change_fmt',
    ],
    aliases=[
        'Neighborhood', 'Borough',
        'COVID Deaths / 100K', 'Total Deaths',
        'Income Tier', 'Poverty Rate',
        '2020 Ridership', 'Ridership Rebound',
    ],
    localize=False,
    sticky=True,
    style='background-color: #ffffff; border: 1px solid #999; border-radius: 4px; box-shadow: 3px 3px 6px rgba(0,0,0,0.25); font-size: 13px; max-width: 420px; white-space: normal;',
)

folium.GeoJson(
    gdf_map2.__geo_interface__,
    style_function=style_fn2,
    highlight_function=highlight_fn2,
    tooltip=tooltip2,
    name='COVID Death Rate',
).add_to(m2)

# ── Subway lines (client-side JS fetch via MacroElement) ──────────────────
from branca.element import MacroElement
from jinja2 import Template
import json as _json

class _SubwayLayer2(MacroElement):
    def __init__(self, line_colors):
        super().__init__()
        self.line_colors_json = _json.dumps(line_colors)
        self._template = Template(
            "{%- macro script(this, kwargs) -%}(function(){var _map={{this._parent.get_name()}};_map.createPane('subwayPane');_map.getPane('subwayPane').style.zIndex='450';setTimeout(function(){fetch('https://data.ny.gov/resource/s692-irgq.geojson?$limit=50000').then(function(r){return r.json();}).then(function(data){var lc={{this.line_colors_json}};L.geoJson(data,{pane:'subwayPane',style:function(f){var s=(f.properties.service||'').trim().toUpperCase();return {color:lc[s]||'#555555',weight:3,opacity:0.8};},onEachFeature:function(f,l){if(f.properties.service_name)l.bindTooltip(f.properties.service_name);}}).addTo(_map);}).catch(function(e){console.warn('Subway fetch failed:',e);});},1000);})();{%- endmacro -%}"
        )

_SubwayLayer2(LINE_COLORS).add_to(m2)


class _StationPane(MacroElement):
    def __init__(self):
        super().__init__()
        self._template = Template(
            "{%- macro script(this, kwargs) -%}"
            "{{this._parent.get_name()}}.createPane('stationsPane');"
            "{{this._parent.get_name()}}.getPane('stationsPane').style.zIndex='550';"
            "{%- endmacro -%}"
        )
_StationPane().add_to(m2)

# ── Station markers ───────────────────────────────────────────────────────
stations_fg2 = folium.FeatureGroup(name='Subway stations', show=True)
for _, row in stations_df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='#333333',
        fill=True,
        fill_color='#ffffff',
        fill_opacity=0.9,
        weight=1.5,
        tooltip=row['station_complex'],
    ).add_to(stations_fg2)

# ── Hover effects ─────────────────────────────────────────────────────────
class _StationHover(MacroElement):
    def __init__(self):
        super().__init__()
        self._template = Template(
            "{%- macro script(this, kwargs) -%}(function(){var _fg={{this._parent.get_name()}};_fg.eachLayer(function(l){l.on('mouseover',function(){this.setRadius(6);this.setStyle({fillColor:'#FCCC0A',fillOpacity:1,color:'#1a1a1a',weight:2.5});});l.on('mouseout',function(){this.setRadius(4);this.setStyle({fillColor:'#ffffff',fillOpacity:0.9,color:'#333333',weight:1.5});});});})();{%- endmacro -%}"
        )
_StationHover().add_to(stations_fg2)
stations_fg2.add_to(m2)

# ── Legend ────────────────────────────────────────────────────────────────
bin_labels = [
    f'{int(bin_edges[i])} – {int(bin_edges[i + 1])}'
    for i in range(5)
]
legend_rows2 = []
for i in range(5):
    color = BIN_COLORS[i]
    label = bin_labels[i]
    legend_rows2.append(
        f'<span style="background:{color};width:14px;height:14px;'
        f'display:inline-block;border:1px solid #ccc;border-radius:2px;'
        f'margin-right:6px;"></span>{label}<br>'
    )
legend_items2 = '\n'.join(legend_rows2)
legend_html2 = (
    '<div style="position:fixed; bottom:30px; left:30px; z-index:1000; '
    'background:white; padding:12px 16px; border-radius:6px; '
    'box-shadow:0 2px 6px rgba(0,0,0,0.3); font-family:sans-serif; '
    'font-size:13px; line-height:1.9;">'
    f'<b>COVID Deaths / 100K</b><br>{legend_items2}'
    '</div>'
)
m2.get_root().html.add_child(folium.Element(legend_html2))

folium.LayerControl().add_to(m2)

# ── Save HTML and inject stationsPane into both the file and inline display ─
import re as _re

m2.save('map_covid_death.html')
with open('map_covid_death.html') as _f: _html = _f.read()
_html = _re.sub(
    r'(L\.circleMarker\(\s*\[[^\]]+\],\s*){',
    r'\1{"pane": "stationsPane", ',
    _html
)
with open('map_covid_death.html', 'w') as _f: _f.write(_html)

from IPython.display import IFrame as _IFrame
_IFrame("map_covid_death.html", width="100%", height=620)

### Map 2: Story Map Narrative

**Cumulative COVID-19 Death Rate per 100,000 Residents
by Neighborhood, New York City (through 2022)**

This map uses the same MODZCTA geography as Map 1, which allows 
direct visual comparison between the two. Deaths are counted at 
residence, where people lived, not where they worked or rode 
the subway.

Death rate is a continuous, directional variable: more is worse. 
A sequential choropleth is the appropriate representation. The 
color runs from white at the lowest burden to dark blood red at 
the highest. The visual argument precedes the legend.

Classification is natural breaks (Jenks), k=5. The death rate 
distribution is right-skewed with a thin but extreme upper tail. 
Natural breaks isolates the four highest-burden neighborhoods 
(1,032–1,613 deaths per 100,000) as their own class without 
compressing the variation in the dense middle range. Equal 
interval would wash that variation out; quantile would 
artificially spread the extreme tail across multiple classes. 
The bin edges, 11–344, 344–569, 569–767, 767–1,032, 
1,032–1,613, emerged from the data.

The pattern is the argument. The darkest neighborhoods are all 
outer borough: East New York (1,612), Brighton Beach/Coney 
Island (1,357), Flushing/Murray Hill (1,181), Brighton 
Beach/Manhattan Beach (1,175), Edgemere/Far Rockaway (1,032). 
The palest are all Manhattan core: Financial District (11), 
Battery Park City (54), Long Island City (52). The gap between 
the highest and lowest is 141 times, in the same city. Working 
poor neighborhoods averaged 662 deaths per 100,000; upper-middle 
class neighborhoods averaged 335. The correlation between death 
rate and median income is -0.61, the strongest in the dataset.

The map darkens in the same places Map 1 showed working poor and 
working class neighborhoods. The subway overlay makes the 
connection explicit, showing how the lines serving the highest-burden 
areas, the J, L, and G in East Brooklyn, the A in the Far Rockaway 
corridor, are the same lines with the most recurring service 
disruptions as of 2026.

This finding is not isolated. NYU Furman Center independently 
identified East New York as the highest death rate neighborhood 
in the city using the same public data. Zhong et al. (2022) 
found the same negative relationship between median household 
income and death rate per capita across the same 177 MODZCTAs.

-------

## Map 3: LISA Cluster Map

Categorical map of Local Moran's I cluster types at p < 0.05 
(Queen contiguity, row-standardized weights). Tooltip includes 
cluster type, death rate, income tier, Local Moran's I 
statistic, p-value, and contiguous neighbor names.

| Color | Type | Meaning |
|-------|------|---------|
| Red | High-High | High burden cluster |
| Blue | Low-Low | Low burden cluster |
| Orange | High-Low | Outlier — high amid low |
| Purple | Low-High | Outlier — low amid high |
| Gray | Not Significant | No detectable spatial pattern |

Saved to `map_lisa.html`.

In [12]:
LISA_COLORS = {
    'High-High':       '#c0392b',
    'Low-Low':         '#2980b9',
    'High-Low':        '#e67e22',
    'Low-High':        '#8e44ad',
    'Not Significant': '#cccccc',
}

LISA_LEGEND_LABELS = {
    'High-High':       'High burden cluster',
    'Low-Low':         'Low burden cluster',
    'High-Low':        'Outlier (high amid low)',
    'Low-High':        'Outlier (low amid high)',
    'Not Significant': 'Not significant',
}

# ── Pre-format display columns ────────────────────────────────────────────
gdf_map3 = gdf.to_crs(epsg=4326).copy()
gdf_map3['lisa_color']           = gdf_map3['lisa_type'].map(LISA_COLORS)
gdf_map3['covid_death_rate_fmt'] = gdf_map3['COVID_DEATH_RATE'].apply(
    lambda x: 'N/A' if pd.isna(x) else f'{x:.1f}'
)

# ── Build map ─────────────────────────────────────────────────────────────
fig3 = folium.Figure(width='100%', height=620)
m3 = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles='CartoDB positron',
).add_to(fig3)

def style_fn3(feature):
    return {
        'fillColor': feature['properties']['lisa_color'],
        'color': 'white',
        'weight': 0.6,
        'fillOpacity': 0.75,
    }

def highlight_fn3(feature):
    return {
        'fillColor': feature['properties']['lisa_color'],
        'color': '#333333',
        'weight': 2.5,
        'fillOpacity': 0.95,
    }

tooltip3 = GeoJsonTooltip(
    fields=[
        'NEIGHBORHOOD_NAME', 'BOROUGH_GROUP',
        'lisa_type', 'covid_death_rate_fmt',
        'income_tier', 'local_i', 'p_value', 'neighbor_names',
    ],
    aliases=[
        'Neighborhood', 'Borough',
        'LISA Cluster', 'COVID Deaths / 100K',
        'Income Tier', "Local Moran's I", 'p-value', 'Contiguous Neighbors',
    ],
    localize=False,
    sticky=True,
    style='background-color: #ffffff; border: 1px solid #999; border-radius: 4px; box-shadow: 3px 3px 6px rgba(0,0,0,0.25); font-size: 13px; max-width: 420px; white-space: normal;',
)

folium.GeoJson(
    gdf_map3.__geo_interface__,
    style_function=style_fn3,
    highlight_function=highlight_fn3,
    tooltip=tooltip3,
    name='LISA Clusters',
).add_to(m3)

# ── Subway lines (client-side JS fetch via MacroElement) ──────────────────
from branca.element import MacroElement
from jinja2 import Template
import json as _json

class _SubwayLayer3(MacroElement):
    def __init__(self, line_colors):
        super().__init__()
        self.line_colors_json = _json.dumps(line_colors)
        self._template = Template(
            "{%- macro script(this, kwargs) -%}(function(){var _map={{this._parent.get_name()}};_map.createPane('subwayPane');_map.getPane('subwayPane').style.zIndex='450';setTimeout(function(){fetch('https://data.ny.gov/resource/s692-irgq.geojson?$limit=50000').then(function(r){return r.json();}).then(function(data){var lc={{this.line_colors_json}};L.geoJson(data,{pane:'subwayPane',style:function(f){var s=(f.properties.service||'').trim().toUpperCase();return {color:lc[s]||'#555555',weight:3,opacity:0.8};},onEachFeature:function(f,l){if(f.properties.service_name)l.bindTooltip(f.properties.service_name);}}).addTo(_map);}).catch(function(e){console.warn('Subway fetch failed:',e);});},1000);})();{%- endmacro -%}"
        )

_SubwayLayer3(LINE_COLORS).add_to(m3)


class _StationPane(MacroElement):
    def __init__(self):
        super().__init__()
        self._template = Template(
            "{%- macro script(this, kwargs) -%}"
            "{{this._parent.get_name()}}.createPane('stationsPane');"
            "{{this._parent.get_name()}}.getPane('stationsPane').style.zIndex='550';"
            "{%- endmacro -%}"
        )
_StationPane().add_to(m3)

# ── Station markers ───────────────────────────────────────────────────────
stations_fg3 = folium.FeatureGroup(name='Subway stations', show=True)
for _, row in stations_df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='#333333',
        fill=True,
        fill_color='#ffffff',
        fill_opacity=0.9,
        weight=1.5,
        tooltip=row['station_complex'],
    ).add_to(stations_fg3)

# ── Hover effects ─────────────────────────────────────────────────────────
class _StationHover(MacroElement):
    def __init__(self):
        super().__init__()
        self._template = Template(
            "{%- macro script(this, kwargs) -%}(function(){var _fg={{this._parent.get_name()}};_fg.eachLayer(function(l){l.on('mouseover',function(){this.setRadius(6);this.setStyle({fillColor:'#FCCC0A',fillOpacity:1,color:'#1a1a1a',weight:2.5});});l.on('mouseout',function(){this.setRadius(4);this.setStyle({fillColor:'#ffffff',fillOpacity:0.9,color:'#333333',weight:1.5});});});})();{%- endmacro -%}"
        )
_StationHover().add_to(stations_fg3)
stations_fg3.add_to(m3)

# ── Legend ────────────────────────────────────────────────────────────────
legend_order3 = ['High-High', 'Low-Low', 'High-Low', 'Low-High', 'Not Significant']
legend_rows3 = []
for k in legend_order3:
    color = LISA_COLORS[k]
    label = LISA_LEGEND_LABELS[k]
    legend_rows3.append(
        f'<span style="background:{color};width:14px;height:14px;'
        f'display:inline-block;border:1px solid #ccc;border-radius:2px;'
        f'margin-right:6px;"></span>{label}<br>'
    )
legend_items3 = '\n'.join(legend_rows3)
legend_html3 = (
    '<div style="position:fixed; bottom:30px; left:30px; z-index:1000; '
    'background:white; padding:12px 16px; border-radius:6px; '
    'box-shadow:0 2px 6px rgba(0,0,0,0.3); font-family:sans-serif; '
    'font-size:13px; line-height:1.9;">'
    f'<b>LISA Cluster Type</b><br>{legend_items3}'
    '</div>'
)
m3.get_root().html.add_child(folium.Element(legend_html3))

folium.LayerControl().add_to(m3)

# ── Save HTML and inject stationsPane into both the file and inline display ─
import re as _re

m3.save('map_lisa.html')
with open('map_lisa.html') as _f: _html = _f.read()
_html = _re.sub(
    r'(L\.circleMarker\(\s*\[[^\]]+\],\s*){',
    r'\1{"pane": "stationsPane", ',
    _html
)
with open('map_lisa.html', 'w') as _f: _f.write(_html)

from IPython.display import IFrame as _IFrame
_IFrame("map_lisa.html", width="100%", height=620)

### Map 3: Story Map Narrative

**Local Spatial Autocorrelation of COVID-19 Death Rate
(LISA Clusters, Queen Contiguity, p < 0.05)**

This map covers all 177 MODZCTAs. Spatial autocorrelation 
requires the full city geography. A partial analysis would 
distort the weights matrix and produce misleading cluster 
classifications.

LISA cluster types are statistically defined categories, not 
points on a scale. A sequential color scheme would imply 
ordering that does not exist between High-High, Low-Low, and 
the outlier types. Five distinct colors carry five distinct 
analytical meanings.

Each neighborhood is assigned a cluster type based on its own 
death rate relative to its contiguous neighbors, tested at 
p < 0.05 across 999 permutations, the standard threshold for 
exploratory spatial analysis. The global result is not 
ambiguous: Moran's I = 0.46, p = 0.001, z = 8.61. 

For reference, with a p value less than 0.05, it means that
anything above a z score of 1.96 is statistically significant. **The z score here is 8.61.**

This, as an understatement, implies that citywide distribution of 
COVID-19 death rates is not random.

76% of neighborhoods return Not Significant, which means the 
burden pattern is concentrated rather than diffuse, the 
clusters that do appear carry real weight.

There are 9 High-High clusters. None are upper-middle class. 
Six are working class, three are working poor. All nine are in 
Brooklyn, Queens, or the Bronx: Brighton Beach/Coney Island 
(1,357), Arverne/Edgemere (789), Bath Beach/Bensonhurst/ 
Gravesend (692), Morris Park/Pelham Bay (650), Canarsie (641), 
Corona/North Corona (640), and two MODZCTAs covering Cypress 
Hills/East New York (631 and 593).

There are 25 Low-Low clusters. Twenty-three of the 25 (92%) 
are upper-middle class, concentrated in the Manhattan core.

The outliers are neighborhoods where the spatial pattern and 
the income geography diverge. 

A High-Low is a high-burden neighborhood surrounded by 
lower-burden neighbors, a hot spot in a cold area, a place 
that bears the weight of the outer boroughs but sits 
geographically isolated from other high-burden zones. 
Chinatown/Lower East Side is the clearest example: 654 deaths 
per 100,000, working poor, in Manhattan, surrounded by neighborhoods 
with substantially lower rates. 

Low-High is the inverse, a relatively protected pocket inside 
a high-burden zone, a neighborhood that bucked the trend of 
the area around it.

The 0 of 9 and 23 of 25 figures are the cleanest numbers in 
this dataset. This map turns the visual observation from Map 2 
into a statistical claim: the burden was not scattered. It was 
geographically structured, and that structure correlates with 
income tier and transit dependence.

-----

## Conclusion

**Storytelling.** The three maps are meant to be read in 
sequence. Map 1 establishes who lives where. Map 2 shows what 
happened to them. Map 3 makes the statistical case that the 
pattern wasn't random. Income tier carries through all three 
maps in the tooltip, so the reader can always connect a death 
rate or a cluster type back to the economic baseline. The 141x 
gap between East New York and the Financial District is the 
story, the maps just show it.

**Visual Hierarchy.** The color choices were deliberate. Map 1 
uses three distinct categorical colors that don't imply ranking. 
Map 2 inverts that approach with a visceral color that
carries meaning, running white into blood red. The reader is
meant to understand the argument before reading the legend. 
Map 3 uses five categorical colors because LISA types aren't ordered; 
the gray mass of non-significant neighborhoods makes the colored 
clusters stand out without me having to say anything. All three 
maps use CartoDB positron so the basemap doesn't compete with 
the data.

**Data Wizardry.** Adding LISA wasn't exactly required. I added it 
because Map 2 raised a spatial question I couldn't answer 
visually. The natural breaks classification came from the data 
distribution. The 54 neighborhoods with no subway station stay 
in all three maps and are labeled in every tooltip; East New York 
is one of them. The Moran's I result (0.46, p = 0.001) shows that this pattern is statistically significant. Where Moran's I = 0 means completely random and Moran's I = 1 means perfect clustering (-1 implies checkerboarding or negative spatial correlation. Not relevant here, but worth noting), a value of 0 would mean death rates are randomly distributed across the city. At 0.46, they are not. 
Additionally, the p-value of 0.001 means this pattern appeared by chance only once in a thousand random reshufflings of the data. 
The ridership rebound I found here agrees with the findings of the NYS 
Comptroller and what the MTA's own reports show, which I didn't know going in.

-----

## Summary

Working poor neighborhoods had roughly double the COVID-19 death 
rate of upper-middle class neighborhoods. The neighborhoods 
where subway ridership held through 2020 are the same 
neighborhoods where death rates were highest. 

That pattern was not random, with Moran's I = 0.46, p = 0.001, and all 9 High-High LISA clusters falling in working class and working poor 
outer-borough neighborhoods, while 23 of 25 Low-Low clusters 
are upper-middle class.

The Financial District recorded 11 deaths per 100,000. East New 
York recorded 1,612. 

This death count disparity is not a statistical mistake. Those lives are what it cost our families to keep New York City running.

---

## References

### Data Sources Used in Analysis

- [NYC Department of Health and Mental Hygiene. COVID-19 Data Repository.](https://github.com/nychealth/coronavirus-data)
  Source for NYC COVID-19 data, including MODZCTA-level cumulative cases, deaths, and rates.

- [NYC DOHMH. MODZCTA COVID-19 data: data-by-modzcta.csv.](https://github.com/nychealth/coronavirus-data/blob/master/totals/data-by-modzcta.csv)
  Direct CSV used for MODZCTA-level COVID outcomes.

- [NYC DOHMH. Modified ZIP Code Tabulation Areas explanation.](https://nychealth.github.io/covid-maps/modzcta-geo/about.html)
  Background on MODZCTA geography and why NYC Health uses it for neighborhood-level health reporting.

- [NYC Open Data. Modified Zip Code Tabulation Areas (MODZCTA).](https://data.cityofnewyork.us/Health/Modified-Zip-Code-Tabulation-Areas-MODZCTA-/pri4-ifjk)
  MODZCTA boundary geography used for mapping.

- [NYC Environmental Health. NYC Geography GitHub Repository.](https://github.com/nycehs/NYC_geography)
  Reference repository for NYC geographic boundary files, including simplified GeoJSON and shapefile resources.

- [MTA / State of New York Open Data. MTA Subway Hourly Ridership: 2020-2024.](https://data.ny.gov/Transportation/MTA-Subway-Hourly-Ridership-2020-2024/wujg-7c2s)
  Source for station-level subway ridership used to calculate 2020 ridership volume and 2020-2022 rebound.

- [MTA / State of New York Open Data. MTA Subway Stations.](https://data.ny.gov/Transportation/MTA-Subway-Stations/39hk-dx4f)
  Source for subway station locations used in the spatial join to MODZCTAs.

- [MTA / State of New York Open Data. MTA Subway Service Lines.](https://data.ny.gov/Transportation/MTA-Subway-Service-Lines/s692-irgq)
  Source for subway line geometries used in the interactive map.

- [U.S. Census Bureau. ACS 2021 5-Year Estimates, Selected Economic Characteristics, DP03.](https://data.census.gov/table/ACSDP5Y2021.DP03)
  Source for median household income and related neighborhood economic variables.

- [U.S. Census Bureau. Census Data API: 2021 ACS 5-Year Profile Variables.](https://api.census.gov/data/2021/acs/acs5/profile/variables.html)
  API documentation used to retrieve ACS 2021 variables.

All of this data is incorporated in either `merged_modzcta.geojson` or the maps.

---

### External Validation and Context

- [New York State Comptroller. Archived Subway Recovery Tracker.](https://www.osc.ny.gov/reports/osdc/subway-recovery-tracker-archived)
  Context for post-2020 subway ridership patterns and recovery differences across the city.

- [New York State Comptroller. Subway Recovery Tracker.](https://www.osc.ny.gov/osdc/subway-recovery-tracker)
  Current version of the subway recovery tracker.

- [New York City Comptroller. Beyond Rush Hour.](https://comptroller.nyc.gov/reports/beyond-rush-hour/)
  Context on changing commute patterns, off-peak transit use, and lower-income riders.

- [New York City Comptroller. Riders Return.](https://comptroller.nyc.gov/reports/riders-return/)
  Context on subway and bus ridership recovery after the first years of the pandemic.

- [New York City Comptroller. New York City's Frontline Workers.](https://comptroller.nyc.gov/reports/new-york-citys-frontline-workers/)
  Source for demographic and labor context on frontline workers during COVID-19.

- [New York City Comptroller. Comptroller Stringer Releases New Data Profiles of Frontline Workers and Calls for Protections Amid COVID-19 Pandemic.](https://comptroller.nyc.gov/newsroom/comptroller-stringer-releases-new-data-profiles-of-frontline-workers-and-calls-for-protections-amid-covid-19-pandemic/)
  Additional context on the scale and composition of NYC's frontline workforce.

- [NYU Furman Center. COVID-19 Cases in New York City, a Neighborhood-Level Analysis.](https://furmancenter.org/thestoop/entry/covid-19-cases-in-new-york-city-a-neighborhood-level-analysis)
  Independent neighborhood-level analysis using the same public data; corroborates the outer-borough death rate pattern and identifies East New York as the highest death rate neighborhood in the city.

- [Bilal et al. Spatial Inequities in COVID-19 Testing, Positivity, Confirmed Cases, and Mortality in 3 U.S. Cities. *Annals of Internal Medicine*, 2021.](https://pmc.ncbi.nlm.nih.gov/articles/PMC8029592/)
  Scholarly context for spatial inequities in COVID-19 outcomes across New York City, Philadelphia, and Chicago.

- [Zhong et al. Neighborhood disparities in COVID-19 outcomes in New York City. *Journal of Urban Health*, 2022.](https://pmc.ncbi.nlm.nih.gov/articles/PMC9042413/)
  Scholarly context on MODZCTA-level neighborhood disparities in NYC COVID-19 outcomes; uses the same 177 MODZCTA geography and finds the same negative relationship between median household income and death rate per capita.

- [PySAL Development Team. esda: Exploratory Spatial Data Analysis.](https://esda.readthedocs.io/en/latest/)
  Python library used for Local Moran's I (LISA) spatial autocorrelation analysis.

---

### News and Current Reporting

- [MTA. Annual Disclosure Statement Update, November 24, 2020.](https://www.mta.info/document/24571)
  MTA disclosure document covering COVID-era operating conditions and service changes.

- [Queens Daily Eagle. MTA suspends overnight subway service indefinitely.](https://queenseagle.com/all/mta-suspends-overnight-subway-service)
  Reporting on the May 2020 overnight subway suspension.

- [Streetsblog NYC. More Than 20 Subway Stations Exceeded Pre-Pandemic Ridership Figures in December.](https://nyc.streetsblog.org/2023/01/10/over-20-subway-stations-exceeded-pre-pandemic-ridership-figures-in-december)
  Reporting on outer-borough ridership recovery; includes MTA leadership statements corroborating higher ridership retention in working-class neighborhoods throughout the pandemic.

- [ABC7NY. G train service disruptions get underway for subway riders.](https://abc7ny.com/post/train-service-disruptions-get-underway-subway-riders-brooklyn-queens/18987619/)
  Reporting on recurring 2026 G train service changes, including track work and signal modernization affecting Brooklyn and Queens.

- [MTA. Service changes on the G line in 2026.](https://www.mta.info/article/service-changes-g-line-2026)
  Official MTA notice for recurring G train service changes in 2026.

- [MTA. MTA Weekender: April 17-20, 2026.](https://www.mta.info/article/mta-weekender-april-17-20-2026)
  Official MTA service-change notice documenting G train suspensions between Court Sq and Bedford-Nostrand Avs for signal modernization, and J train suspensions between Jamaica Center-Parsons/Archer and Crescent St for track replacement.

- [MTA. MTA Weekender: April 10-13, 2026.](https://www.mta.info/article/mta-weekender-april-10-13-2026)
  Official MTA service-change notice documenting J train suspensions between Jamaica Center-Parsons/Archer and Crescent St, with J90 shuttle buses serving stops between 121 St and Crescent St.

- [MTA. MTA Weekender: February 20-23, 2026.](https://www.mta.info/article/mta-weekender-february-20-23-2026)
  Official MTA service-change notice documenting J and M service changes between Marcy Av and Myrtle Av for structural work.

- [MTA. MTA Weekender: Martin Luther King Jr. Day Weekend 2026.](https://www.mta.info/article/mta-weekender-martin-luther-king-jr-day-weekend-2026)
  Official MTA service-change notice documenting J train suspensions between Jamaica Center-Parsons/Archer and Crescent St and L train suspensions between Lorimer St and Broadway Junction.

- [MTA. Major service changes on the 4 and 5 lines in January and February 2026.](https://www.mta.info/article/major-service-changes-4-and-5-lines-january-and-february-2026)
  Official MTA notice documenting 4 and 5 line service changes for switch replacement in January and February 2026.

---

### Background and Framing

- [New York City Comptroller. The Human Cost of Subway Delays.](https://comptroller.nyc.gov/reports/the-human-cost-of-subway-delays-a-survey-of-new-york-city-riders/)
  Background on how subway delays affect lower-income New Yorkers.

- [New York City Comptroller. Comptroller Stringer Releases New Report and Survey Results on the Human Impacts of Subway Delays.](https://comptroller.nyc.gov/newsroom/comptroller-stringer-releases-new-report-and-survey-results-on-the-human-impacts-of-subway-delays-on-straphangers-lives/)
  Related press release summarizing unequal impacts of subway delays.

- [NYC Open Data. Open Data for All New Yorkers.](https://opendata.cityofnewyork.us/)
  General reference for NYC's open data infrastructure.